# Hands-On Activity: Automation with GitHub Actions
### Write real CI/CD workflows for an ML project and watch them run

**Scenario:** The churn project works, but everything is still done by hand. You are going to
automate it: tests on every pull request, a quality gate that blocks a model whose recall drops,
and a weekly retraining job with a manual override button.

**Agenda**

1. Setup
2. Step 1 — Your first CI workflow (**your code**)
3. Step 2 — A matrix, and trigger filtering (**your code**)
4. Step 3 — An ML quality gate that blocks a bad model (**your code**)
5. Step 4 — Scheduled retraining with a manual button (**your code**)
6. Step 5 — Secrets, artifacts, and steps that always run (**your code**)
7. Step 6 — Push it to a real GitHub repository
8. Discussion and extensions

### Learning objectives
By the end of this activity you will be able to:
- Write a valid GitHub Actions workflow: triggers, jobs, steps, `uses` versus `run`
- Use a matrix to run one job across several configurations
- Make a job fail on purpose with a non-zero exit code, and use that as a quality gate
- Wire up `needs` so a failed gate stops everything downstream
- Schedule a workflow with cron and add a `workflow_dispatch` button with inputs
- Reference secrets safely and keep files with artifacts
- Push a workflow to GitHub and read the result in the Actions tab


## Setup

Run these cells as-is. They create a small but realistic ML repository to automate, and install a
local workflow runner so you get results in seconds instead of pushing to GitHub and waiting.

**About the local runner.** It is a teaching tool, not a clone of GitHub. It does execute your
`run:` steps for real, expand `strategy.matrix`, respect `needs`, honour `if: always()`, mask
secrets, and skip workflows whose trigger does not match. It does **not** download real actions,
so `uses:` steps are reported as simulated, and it supports only a small subset of expressions.
The YAML you write is genuine GitHub Actions YAML, and in Step 6 you will run it on GitHub for
real to prove it.


In [ ]:
%pip install -q pyyaml pytest scikit-learn pandas


In [ ]:
import os, pathlib, shutil, textwrap, subprocess, json
import pandas as pd

if "BASE" not in globals():
    BASE = pathlib.Path.cwd()
os.chdir(BASE)

LAB = BASE / "gha-lab"
PROJECT = LAB / "churn-ci"

if LAB.exists():
    shutil.rmtree(LAB)
(PROJECT / ".github" / "workflows").mkdir(parents=True)
(PROJECT / "src").mkdir()
(PROJECT / "tests").mkdir()
(PROJECT / "data").mkdir()
os.chdir(PROJECT)
print("Project folder:", PROJECT)


### The project you will automate

A trimmed-down version of the churn project: a training script, a evaluation script that writes
`metrics.json`, a gate script that reads it, and one unit test.


In [ ]:
def write(path, text):
    p = pathlib.Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(textwrap.dedent(text).lstrip())
    return p


write("requirements.txt", """
    scikit-learn
    pandas
    pytest
""")

write("src/make_data.py", """
    import numpy as np, pandas as pd, sys
    rng = np.random.default_rng(int(sys.argv[1]) if len(sys.argv) > 1 else 42)
    n = 3000
    tenure = rng.gamma(2.0, 12.0, n).clip(0, 72)
    charges = rng.normal(70, 25, n).clip(15, 200)
    tickets = rng.poisson(1.2, n).clip(0, 15)
    z = -0.08*tenure + 0.04*charges + 0.7*tickets - 3.0
    churned = (rng.uniform(0, 1, n) < 1/(1+np.exp(-z))).astype(int)
    pd.DataFrame({"tenure_months": tenure.round(3), "monthly_charges": charges.round(3),
                  "num_support_tickets": tickets, "churned": churned}).to_csv("data/train.csv", index=False)
    print("wrote data/train.csv", n, "rows, churn rate", round(churned.mean(), 3))
""")

write("src/train_eval.py", """
    import json, argparse, pandas as pd
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import roc_auc_score, recall_score

    p = argparse.ArgumentParser()
    p.add_argument("--threshold", type=float, default=0.5)
    args = p.parse_args()

    FEATURES = ["tenure_months", "monthly_charges", "num_support_tickets"]
    df = pd.read_csv("data/train.csv")
    X_tr, X_te, y_tr, y_te = train_test_split(
        df[FEATURES], df["churned"], test_size=0.25, stratify=df["churned"], random_state=42)

    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    model.fit(X_tr, y_tr)
    proba = model.predict_proba(X_te)[:, 1]
    metrics = {
        "auc": round(float(roc_auc_score(y_te, proba)), 4),
        "recall": round(float(recall_score(y_te, (proba >= args.threshold).astype(int))), 4),
    }
    json.dump(metrics, open("metrics.json", "w"), indent=2)
    print("metrics:", metrics)
""")

write("src/check_metrics.py", """
    import json, sys, argparse
    p = argparse.ArgumentParser()
    p.add_argument("--min-recall", type=float, required=True)
    args = p.parse_args()

    m = json.load(open("metrics.json"))
    print(f"recall = {m['recall']:.4f}, required >= {args.min_recall}")
    if m["recall"] < args.min_recall:
        print("QUALITY GATE FAILED: recall is below the required minimum")
        sys.exit(1)
    print("Quality gate passed")
""")

write("tests/test_data.py", """
    import pandas as pd

    def test_columns_present():
        df = pd.read_csv("data/train.csv")
        for col in ["tenure_months", "monthly_charges", "num_support_tickets", "churned"]:
            assert col in df.columns

    def test_target_is_binary():
        df = pd.read_csv("data/train.csv")
        assert set(df["churned"].unique()) <= {0, 1}
""")

subprocess.run("python src/make_data.py", shell=True)
print(sorted(str(p) for p in pathlib.Path(".").rglob("*.py")))


### The local runner and the workflow helper

`run_workflow(path, event=..., inputs=...)` executes a workflow file.
`write_workflow(path, text)` writes a workflow and immediately checks that it is valid YAML, so a
typo is caught here rather than three steps later.


In [ ]:
RUNNER_SOURCE = r'''
"""A small local runner for GitHub Actions workflow files (teaching tool)."""
import os, re, sys, stat, shutil, tempfile, subprocess, itertools, yaml

_SHIM_DIR = None


def _local_env():
    """Make authentic workflow YAML runnable on any machine.

    On a GitHub runner `python` and `pip` always exist and pip may install
    freely. Locally that is not guaranteed, so we (a) provide a `python`
    shim pointing at this interpreter if the name is missing, and (b) allow
    pip to install on externally managed Python installations.
    """
    global _SHIM_DIR
    env = dict(os.environ)
    env.setdefault("PIP_BREAK_SYSTEM_PACKAGES", "1")
    env.setdefault("PIP_DISABLE_PIP_VERSION_CHECK", "1")
    if shutil.which("python") is None and os.name != "nt":
        if _SHIM_DIR is None:
            _SHIM_DIR = tempfile.mkdtemp(prefix="gha-shim-")
            shim = os.path.join(_SHIM_DIR, "python")
            with open(shim, "w") as f:
                f.write('#!/bin/sh\nexec "%s" "$@"\n' % sys.executable)
            os.chmod(shim, os.stat(shim).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
        env["PATH"] = _SHIM_DIR + os.pathsep + env.get("PATH", "")
    return env

EXPR = re.compile(r"\$\{\{\s*([^}]+?)\s*\}\}")


def _resolve(text, ctx):
    if not isinstance(text, str):
        return text
    def sub(m):
        expr = m.group(1).strip()
        for prefix in ("matrix.", "env.", "inputs.", "github."):
            if expr.startswith(prefix):
                return str(ctx.get(prefix[:-1], {}).get(expr[len(prefix):], ""))
        if expr.startswith("secrets."):
            return "***"          # GitHub masks secrets in logs; so do we
        return m.group(0)
    return EXPR.sub(sub, text)


def _expand_matrix(job):
    matrix = (job.get("strategy") or {}).get("matrix")
    if not matrix:
        return [{}]
    keys = list(matrix.keys())
    return [dict(zip(keys, combo)) for combo in itertools.product(*(matrix[k] for k in keys))]


def run_workflow(path, event="push", inputs=None, cwd=None):
    wf = yaml.safe_load(open(path))
    # NOTE: PyYAML reads the bare key `on` as the boolean True (see the pitfalls slide)
    triggers = wf.get("on", wf.get(True, {}))
    if isinstance(triggers, str):
        triggers = {triggers: None}
    if isinstance(triggers, list):
        triggers = {t: None for t in triggers}

    if event not in triggers:
        print(f"[skipped] '{wf.get('name', path)}' does not trigger on '{event}'")
        print(f"          triggers defined: {sorted(triggers)}")
        return {"triggered": False, "ok": True, "jobs": {}}

    wf_env = wf.get("env") or {}
    results, overall_ok = {}, True
    print(f"Workflow: {wf.get('name', path)}   (event: {event})")
    print("=" * 66)

    for job_id, job in (wf.get("jobs") or {}).items():
        needs = job.get("needs") or []
        if isinstance(needs, str):
            needs = [needs]
        if any(not results.get(n, {}).get("ok", False) for n in needs):
            print(f"\nJob: {job_id}   [skipped - a job it needs did not succeed]")
            results[job_id] = {"ok": False, "skipped": True}
            overall_ok = False
            continue

        job_ok = True
        for combo in _expand_matrix(job):
            label = job_id + (f"  {combo}" if combo else "")
            print(f"\nJob: {label}   runs-on: {job.get('runs-on', 'ubuntu-latest')}")
            print("-" * 66)
            ctx = {"matrix": combo, "env": {**wf_env, **(job.get("env") or {})},
                   "inputs": inputs or {}, "github": {"event_name": event}}
            failed = False
            for step in job.get("steps", []):
                name = _resolve(step.get("name", step.get("uses", step.get("run", "step"))), ctx)
                cond = str(step.get("if", ""))
                if failed and "always()" not in cond:
                    print(f"  [skipped] {name}")
                    continue
                if "uses" in step:
                    print(f"  [action ] {name}   (simulated locally)")
                    continue
                cmd = _resolve(step["run"], ctx)
                env = {**_local_env(),
                       **{k: str(_resolve(v, ctx)) for k, v in ctx["env"].items()},
                       **{k: str(_resolve(v, ctx)) for k, v in (step.get("env") or {}).items()}}
                print(f"  [run    ] {name}")
                for line in cmd.strip().splitlines():
                    if line.strip():
                        print(f"            $ {line.strip()}")
                p = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd, env=env)
                for line in (p.stdout + p.stderr).strip().splitlines()[:20]:
                    print(f"            | {line}")
                if p.returncode != 0:
                    print(f"  [FAILED ] {name}   (exit code {p.returncode})")
                    failed = True
                else:
                    print(f"  [ok     ] {name}")
            job_ok = job_ok and not failed
        results[job_id] = {"ok": job_ok}
        overall_ok = overall_ok and job_ok

    print("\n" + "=" * 66)
    print("RESULT:", "success" if overall_ok else "FAILURE")
    return {"triggered": True, "ok": overall_ok, "jobs": results}
'''

pathlib.Path("mini_actions.py").write_text(RUNNER_SOURCE)

import yaml
from mini_actions import run_workflow


def write_workflow(path, text):
    """Write a workflow file and check that it parses as YAML."""
    p = write(path, text)
    try:
        parsed = yaml.safe_load(p.read_text())
    except yaml.YAMLError as e:
        print("INVALID YAML -- fix this before running:\n", e)
        return None
    triggers = parsed.get("on", parsed.get(True))
    print(f"Wrote {path}")
    print(f"  name     : {parsed.get('name')}")
    print(f"  triggers : {sorted(triggers) if isinstance(triggers, dict) else triggers}")
    print(f"  jobs     : {list((parsed.get('jobs') or {}).keys())}")
    return parsed


print("Runner ready.")


## Step 1 — Your First CI Workflow

**Your task:** write `.github/workflows/ci.yml` so that it:

1. is named `CI`
2. triggers on `pull_request`, and on `push` to the `main` branch only
3. has one job with id `test` that runs on `ubuntu-latest`
4. has these steps, in this order:
   - `uses: actions/checkout@v4`
   - `uses: actions/setup-python@v5` with `python-version: "3.11"`
   - a `run` step named `Install dependencies` that runs `pip install -r requirements.txt`
   - a `run` step named `Run tests` that runs `python -m pytest -q`

Fill in the YAML string below. Indentation matters: two spaces per level, and list items start
with `- `.


In [ ]:
write_workflow(".github/workflows/ci.yml", """
    name: ___

    on:
      ___:
      ___:
        branches: [___]

    jobs:
      ___:
        runs-on: ___
        steps:
          - uses: ___
          - uses: actions/setup-python@v5
            with:
              python-version: "3.11"
          - name: Install dependencies
            run: ___
          - name: Run tests
            run: ___
""")


Now run it. The `uses:` steps are reported as simulated; the `run:` steps execute for real.


In [ ]:
run_workflow(".github/workflows/ci.yml", event="pull_request")


**Quick check:** On a real runner, what would happen if you deleted the `actions/checkout` step?
Why does that step have to come first?


## Step 2 — A Matrix, and Trigger Filtering

Two changes to the same workflow.

**Your task:**
1. Add a `strategy.matrix` to the `test` job with `python-version: ["3.10", "3.11", "3.12"]`
2. Use `${{ matrix.python-version }}` as the value of `python-version:` in the setup-python step
3. Add a `run` step named `Show version` that echoes which version this copy of the job is using

Then run it and count how many times the job executes.


In [ ]:
write_workflow(".github/workflows/ci.yml", """
    name: CI

    on:
      pull_request:
      push:
        branches: [main]

    jobs:
      test:
        runs-on: ubuntu-latest
        strategy:
          matrix:
            ___: [___, ___, ___]
        steps:
          - uses: actions/checkout@v4
          - uses: actions/setup-python@v5
            with:
              python-version: ${{ ___ }}
          - name: Show version
            run: echo "this job copy is for Python ${{ ___ }}"
          - name: Install dependencies
            run: pip install -r requirements.txt
          - name: Run tests
            run: python -m pytest -q
""")


Run it twice: once with an event it listens for, and once with an event it does not.


In [ ]:
run_workflow(".github/workflows/ci.yml", event="pull_request")

print("\n\n### Now an event this workflow does not listen for:\n")
run_workflow(".github/workflows/ci.yml", event="schedule")


**Quick check:** The job ran three times. If you added a second matrix dimension with two
operating systems, how many copies would run? What does that do to your bill on a private
repository?


## Step 3 — An ML Quality Gate That Blocks a Bad Model

This is the heart of ML CI. You will add a second job that trains, evaluates, and then **fails on
purpose** when recall is too low.

**Your task:** add a job with id `train` that:
1. uses `needs: test`, so it only runs if the tests passed
2. checks out the code and installs dependencies
3. runs `python src/make_data.py` then `python src/train_eval.py`
4. has a step named `Quality gate` that runs
   `python src/check_metrics.py --min-recall 0.60`

The threshold of 0.60 is deliberately higher than this model achieves, so you can watch the gate
do its job.


In [ ]:
write_workflow(".github/workflows/ci.yml", """
    name: CI

    on:
      pull_request:
      push:
        branches: [main]

    jobs:
      test:
        runs-on: ubuntu-latest
        steps:
          - uses: actions/checkout@v4
          - name: Install dependencies
            run: pip install -r requirements.txt
          - name: Run tests
            run: python -m pytest -q

      ___:                      # job id: train
        runs-on: ubuntu-latest
        ___: test                # only run if the test job succeeded
        steps:
          - uses: actions/checkout@v4
          - name: Install dependencies
            run: pip install -r requirements.txt
          - name: Build the dataset
            run: ___
          - name: Train and evaluate
            run: ___
          - name: Quality gate
            run: ___
""")


Run it. Expect `RESULT: FAILURE` — that is the gate working, not a mistake.


In [ ]:
result = run_workflow(".github/workflows/ci.yml", event="pull_request")
print("\nWorkflow passed?", result["ok"])
print("On GitHub this would show a red X on the pull request and block the merge button.")


**Your task:** now imagine the team agrees 0.45 is the right bar for this model. Change the
threshold in the `Quality gate` step to `0.45` and run it again. The whole workflow should pass.

(Only the one line changes, so the cell below rewrites the file with the new value.)


In [ ]:
wf = pathlib.Path(".github/workflows/ci.yml")
# TODO: replace the 0.60 threshold with 0.45
# wf.write_text(wf.read_text().replace(___, ___))

result = run_workflow(".github/workflows/ci.yml", event="pull_request")
print("\nWorkflow passed?", result["ok"])


**Quick check:** The gate works because `check_metrics.py` calls `sys.exit(1)`. Why is a non-zero
exit code enough to stop the whole pipeline, without any special GitHub Actions feature? And what
would have happened to a `deploy` job that declared `needs: train`?


## Step 4 — Scheduled Retraining With a Manual Button

**Your task:** write a second workflow, `.github/workflows/retrain.yml`, that:

1. is named `Weekly retraining`
2. runs on a `schedule` with cron `"0 3 * * 1"` (Mondays at 03:00 UTC)
3. **also** offers `workflow_dispatch` with one input called `reason`
4. has a job `retrain` that installs dependencies, rebuilds the data, trains, and announces the
   reason with `echo`

Watch the quoting on the announce step: a value containing `: ` must be quoted, or the YAML will
not parse. The safest form is a block scalar:

```
- name: Announce
  run: |
    echo "Retraining because: ${{ inputs.reason }}"
```


In [ ]:
write_workflow(".github/workflows/retrain.yml", """
    name: ___

    on:
      ___:
        - cron: "___"
      ___:
        inputs:
          ___:
            description: Why are you retraining?
            default: weekly schedule

    jobs:
      retrain:
        runs-on: ubuntu-latest
        steps:
          - uses: actions/checkout@v4
          - name: Announce
            run: |
              echo "Retraining because: ${{ ___ }}"
          - name: Install dependencies
            run: pip install -r requirements.txt
          - name: Rebuild data and retrain
            run: |
              python src/make_data.py 7
              python src/train_eval.py
""")


Run it both ways: as the weekly schedule, and as a human clicking the button with a reason.


In [ ]:
run_workflow(".github/workflows/retrain.yml", event="schedule",
             inputs={"reason": "weekly schedule"})

print("\n\n### Same workflow, triggered manually with a reason:\n")
run_workflow(".github/workflows/retrain.yml", event="workflow_dispatch",
             inputs={"reason": "drift alert fired on Tuesday"})


**Quick check:** Why is it useful to have both triggers on one workflow instead of duplicating the
file? And why does the lecture recommend that a scheduled retraining job **opens a pull request**
rather than deploying the new model by itself?


## Step 5 — Secrets, Artifacts, and Steps That Always Run

Three finishing touches to `retrain.yml`.

**Your task:**
1. Add a step named `Publish` that sets an env var `TOKEN` to `${{ secrets.REGISTRY_TOKEN }}` and
   runs `echo "publishing with token $TOKEN"`. Watch what the runner prints.
2. Add a step that deliberately fails: `run: exit 1`
3. After it, add a step named `Save metrics` that uses `actions/upload-artifact@v4` **and** has
   `if: always()`, so the artifact is kept even though the previous step failed


In [ ]:
write_workflow(".github/workflows/retrain.yml", """
    name: Weekly retraining

    on:
      schedule:
        - cron: "0 3 * * 1"
      workflow_dispatch:
        inputs:
          reason:
            description: Why are you retraining?
            default: weekly schedule

    jobs:
      retrain:
        runs-on: ubuntu-latest
        steps:
          - uses: actions/checkout@v4
          - name: Install dependencies
            run: pip install -r requirements.txt
          - name: Rebuild data and retrain
            run: |
              python src/make_data.py 7
              python src/train_eval.py
          - name: Publish
            ___:
              TOKEN: ${{ ___ }}
            run: echo "publishing with token $TOKEN"
          - name: Something breaks
            run: ___
          - name: Save metrics
            ___: always()
            uses: actions/upload-artifact@v4
            with:
              name: metrics
              path: metrics.json
""")


In [ ]:
run_workflow(".github/workflows/retrain.yml", event="workflow_dispatch",
             inputs={"reason": "testing secrets and artifacts"})


**Quick check:** Look at the `Publish` step's output. The token printed as `***`. Why does GitHub
mask it, and why is masking still not a good reason to print secrets on purpose? Also: which step
ran after the failure, and which one was skipped?


## Step 6 — Push It to a Real GitHub Repository

Everything so far ran locally. Now prove the YAML is real by letting GitHub run it. This part
happens in your browser and terminal, not in this notebook.

Your project folder is printed below. It already contains `.github/workflows/`, the source, and
the tests.

**Checklist**

1. On GitHub, create a new **empty** repository (no README), for example `churn-ci-demo`. A public
   repository gets unlimited free Actions minutes.
2. In a terminal, from the project folder printed below:
   ```
   git init -b main
   git add .
   git commit -m "ML project with CI workflows"
   git remote add origin https://github.com/<your-username>/churn-ci-demo.git
   git push -u origin main
   ```
3. Open the repository on GitHub and click the **Actions** tab. Your `CI` workflow should already
   be running, because pushing to `main` is one of its triggers.
4. Click into the run. Expand the `test` job, then a step, and compare the log with what the local
   runner printed. Note what is different: the real runner actually downloads `actions/checkout`
   and installs Python.
5. Find `Weekly retraining` in the left sidebar. Because it has `workflow_dispatch`, there is a
   **Run workflow** button. Click it, type a reason, and watch it run.
6. Open a pull request that raises the gate threshold back to `0.60` and watch the check fail on
   the pull request itself.

**Add a status badge** to your README so the result is visible from the front page:
```
![CI](https://github.com/<your-username>/churn-ci-demo/actions/workflows/ci.yml/badge.svg)
```


In [ ]:
print("Push this folder to GitHub:")
print(" ", PROJECT)
print()
print("Files that will go with it:")
for p in sorted(pathlib.Path(".").rglob("*")):
    if p.is_file() and ".git" not in str(p) and "__pycache__" not in str(p):
        print("  ", p)


**Quick check:** Your local runner said a workflow was skipped when the event did not match. On
GitHub, what decides which workflows run when you push? Where in the interface would you look to
confirm a workflow was skipped rather than failed?


### Discussion Questions

1. Your quality gate blocks merges when recall drops. A teammate says it is annoying and wants it
   removed. What would you propose instead, and what from the lecture supports your answer?
2. Training in CI here takes seconds. If it took forty minutes on the real dataset, name three
   things you could change so the pipeline stays useful.
3. The `train` job repeats `checkout` and `Install dependencies` from the `test` job. Why is that
   repetition necessary, and which action would you add to make it cheap?
4. Which parts of this pipeline would need a secret, and which would work fine on a pull request
   opened from a fork?

### If you finish early
- **Add caching** to speed up installs, keyed on a hash of `requirements.txt`.
- **Add a deploy job** that runs only on `main`, needs `train`, and echoes a promotion command.
- **Compare against a baseline** instead of a fixed threshold: commit a `baseline.json` and make
  the gate fail if recall drops more than 0.02 below it.
- **Add `concurrency`** so a new commit cancels the previous in-progress run.
